<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/refactor%2Fm1-audit-normalization/notebooks/m1_pipeline_thu_thap_va_kiem_toan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Mở trong Colab"/></a>

# Quy trình M1: thu thập, kiểm định và chuẩn hóa dữ liệu

Notebook thực hiện một quy trình trên danh sách ID của thành viên: cấu hình phạm vi → thu thập arXiv → thu thập Semantic Scholar → kiểm định bằng chứng → tạo bảng làm việc. Hai pha thu thập lưu điểm khôi phục riêng; kiểm định lưu báo cáo tiến độ; chuẩn hóa tạo lại bảng từ bằng chứng đã lưu. Kiểm định đọc điểm khôi phục và archive arXiv/S2 trên Drive, khôi phục archive vào ổ cục bộ, rồi ghi báo cáo và tiến độ kiểm định lên Drive; không sửa corpus, archive nguồn hay điểm khôi phục arXiv/S2.

Chạy các ô theo thứ tự. Ô cấu hình yêu cầu Colab CPU, MSSV hợp lệ và hai Secret `KAGGLE_API_TOKEN`, `SEMANTIC_SCHOLAR_API_KEY`. Đặt `COHORT_LIMIT = 1` để chạy thử 100 ID; đặt `None` để chạy đủ 5.000 ID. Phạm vi chạy thử và toàn dải dùng thư mục Drive riêng.

Nếu ô báo lỗi hoặc bị ngắt, đọc thông báo và đường dẫn checkpoint. Sau khi xác nhận không còn phiên khác ghi cùng phạm vi, chạy lại ô tương ứng với cùng MSSV, giới hạn và thư mục Drive. Bước kiểm định ghi nhận checkpoint còn thiếu. Bước chuẩn hóa chỉ tạo hàng từ dữ liệu đã đọc và xác minh được.

## Ô 1 — Cấu hình Colab và phạm vi

Ô này kiểm tra CPU, MSSV và hai Secret; gắn Drive; tải repo, chuyển tới SHA runtime đã ghim; cài gói cần thiết; rồi in phạm vi ID, dấu vân tay và đường dẫn lưu.

Nếu báo kernel đã nạp `src`, khởi động lại runtime rồi chạy ô này trước. Nếu báo thiếu Secret, thêm Secret tương ứng trong Colab và cấp quyền truy cập. Nếu MSSV hoặc SHA không hợp lệ, sửa cấu hình rồi chạy lại. Lỗi gắn Drive, tải mã hoặc cài gói cần được xử lý trước khi chạy các pha.

In [ ]:
# SHA mã nguồn dùng cho quy trình M1.
import os
from pathlib import Path
import re
import subprocess
import sys

M1_RUNTIME_COMMIT = "f4b81e26ef2f5dec8a0a73fa7a79abf34534fe9c"
MSSV = ""  # Nhập MSSV trong config/group10_id_ranges.json.
COHORT_LIMIT = None  # None: 50 cohort; 1..50: số cohort đầu tiên.
PILOT_ROOT = COHORT_LIMIT is not None
DRIVE_BASE = Path("/content/drive/MyDrive/M1-pipeline-optimization-pilot" if PILOT_ROOT else "/content/drive/MyDrive/M1-pipeline-optimization-full")
LOCAL_BASE = Path("/content/m1-pipeline-optimization-pilot-local" if PILOT_ROOT else "/content/m1-pipeline-optimization-local")

loaded_src = sorted(name for name in sys.modules if name == "src" or name.startswith("src."))
if loaded_src:
    raise RuntimeError("Kernel đã nạp src trước khi checkout. Chọn Runtime > Restart session rồi chạy lại cell này.")
if os.environ.get("COLAB_GPU") == "1" or os.environ.get("COLAB_TPU_ADDR"):
    raise RuntimeError("Chọn runtime chỉ dùng CPU trong Runtime settings.")
if not MSSV.strip():
    raise ValueError("Nhập MSSV trước khi tiếp tục.")

try:
    from google.colab import userdata
except Exception:
    raise RuntimeError("Notebook M1 cần chạy trên Colab để đọc Colab Secrets.") from None


def _read_required_m1_secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        raise RuntimeError(
            f"Không đọc được Colab Secret `{name}`. Hãy cấp quyền truy cập Secret cho notebook."
        ) from None
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"Thiếu hoặc rỗng Colab Secret bắt buộc `{name}`.")
    return value.strip()


for _secret_name in ("KAGGLE_API_TOKEN", "SEMANTIC_SCHOLAR_API_KEY"):
    _secret_value = _read_required_m1_secret(_secret_name)
del _secret_name, _secret_value

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/Arxiv-Project-Group10")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if not re.fullmatch(r"[0-9a-f]{40}", M1_RUNTIME_COMMIT):
    raise RuntimeError("M1_RUNTIME_COMMIT phải có đủ 40 ký tự SHA.")
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", M1_RUNTIME_COMMIT], check=True)
current_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if current_head != M1_RUNTIME_COMMIT:
    raise RuntimeError("Mã runtime không khớp SHA đã ghim.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.m1_workflow import build_member_selection
SELECTION = build_member_selection(MSSV, COHORT_LIMIT)
DRIVE_ROOT = DRIVE_BASE / MSSV
LOCAL_ROOT = LOCAL_BASE / MSSV
print(f"Runtime: {current_head}")
print("Chiến lược S2: batch_m1")
print(f"MSSV: {MSSV}; IDs: {SELECTION['expected_count']}; cohorts: {len(SELECTION['cohorts'])}")
print(f"Selection fingerprint: {SELECTION['fingerprint']}")
print(f"Drive root: {DRIVE_ROOT}; local root: {LOCAL_ROOT}")


## Ô 2 — Thu thập arXiv

Ô này xác minh ID bằng Atom, lập danh sách các lần nộp và tải mã nguồn. Pha yêu cầu `KAGGLE_API_TOKEN` cùng snapshot và index Kaggle hợp lệ. Kết quả và điểm khôi phục được lưu theo phạm vi đã chọn.

HTTP 429 có thể được tiếp tục tự động trong giới hạn đã cấu hình. HTTP 403, lỗi Kaggle, lỗi điểm khôi phục hoặc hết ngân sách làm ô dừng. Sau khi xử lý nguyên nhân, chạy lại ô với cùng cấu hình và thư mục Drive.

In [ ]:
from src.m1_workflow import run_m1_arxiv_member

KAGGLE_API_TOKEN = _read_required_m1_secret("KAGGLE_API_TOKEN")
try:
    ARXIV_RESULT = run_m1_arxiv_member(
        MSSV,
        drive_root=DRIVE_ROOT,
        local_root=LOCAL_ROOT,
        cohort_limit=COHORT_LIMIT,
        kaggle_token=KAGGLE_API_TOKEN,
        auto_resume_429=True,
        auto_resume_429_max_per_cohort=1,
        auto_resume_429_max_per_session=2,
        auto_resume_429_cooldown_min_seconds=300,
        auto_resume_429_cooldown_max_seconds=600,
    )
finally:
    KAGGLE_API_TOKEN = None

print({key: ARXIV_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "source_complete_papers", "stopped", "stop_reason", "m1_complete", "kaggle", "auto_resume_429",
)})


## Ô 3 — Thu thập Semantic Scholar

Ô này lấy metadata và references cho cùng danh sách ID bằng `batch_m1`. Client kiểm tra định danh; khi batch có lỗi định danh, hệ thống chia nhỏ batch để cô lập ID. ID còn lỗi sau truy vấn riêng vẫn được giữ ở trạng thái lỗi. Bài chưa có bằng chứng references đầy đủ được tiếp tục bằng phân trang; references đã hoàn tất trong checkpoint được giữ nguyên khi chỉ lấy lại metadata. Ba trường quan hệ tùy chọn `contexts`, `intents`, `isInfluential` không thuộc phạm vi dữ liệu M1.

Pha yêu cầu `SEMANTIC_SCHOLAR_API_KEY`. HTTP 429 có thể được tiếp tục tự động trong giới hạn đã cấu hình. HTTP 403, lỗi xác minh checkpoint hoặc hết ngân sách làm ô dừng. Khi phân trang kết thúc nhưng số lượng vẫn lệch, dữ liệu được giữ và cảnh báo được ghi vào `quality.json`.

In [ ]:
from src.m1_workflow import run_m1_s2_member

S2_API_KEY = _read_required_m1_secret("SEMANTIC_SCHOLAR_API_KEY")
try:
    S2_RESULT = run_m1_s2_member(
        MSSV,
        drive_root=DRIVE_ROOT,
        local_root=LOCAL_ROOT,
        semantic_scholar_api_key=S2_API_KEY,
        cohort_limit=COHORT_LIMIT,
        reference_strategy="batch_m1",
        auto_resume_429=True,
        auto_resume_429_max_per_cohort=1,
        auto_resume_429_max_per_session=2,
        auto_resume_429_cooldown_min_seconds=300,
        auto_resume_429_cooldown_max_seconds=600,
    )
finally:
    S2_API_KEY = None

S2_AUTO_RESUME = S2_RESULT.get("auto_resume_429", {})
S2_RESULT_SUMMARY = {key: S2_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "metadata_complete", "references_complete", "stopped", "stop_reason", "m1_complete",
)}
S2_RESULT_SUMMARY["auto_resume_429"] = {key: S2_AUTO_RESUME.get(key) for key in (
    "resume_attempts", "cooldown_seconds", "events_path", "events_sha256",
)}
S2_RESULT_SUMMARY["reference_quality"] = {
    key: value for key, value in S2_RESULT.get("reference_quality", {}).items() if key != "papers"
}
print(S2_RESULT_SUMMARY)


## Ô 4 — Kiểm định và tạo bảng làm việc

Ô này kiểm tra điểm khôi phục arXiv/S2 và archive nguồn, in tiến độ theo lô, rồi ghi báo cáo và tiến độ kiểm định lên Drive sau mỗi lô hoàn tất. Corpus được khôi phục vào ổ cục bộ. Kiểm định không gọi API, không sửa corpus, archive nguồn hoặc điểm khôi phục arXiv/S2, và không nhập dữ liệu Semantic Scholar vào corpus. Khi kết thúc với `status = "completed"`, ô này tạo bảng làm việc. Nếu bị dừng hoặc hết ngân sách, tiến độ được lưu để tiếp tục; bước chuẩn hóa chỉ chạy sau khi kiểm định hoàn tất. Ô này không xuất bảng chính thức.

In [ ]:
from src.m1_workflow import run_m1_validation_member, run_m1_normalization_member

VALIDATION_RESULT = run_m1_validation_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    cohort_limit=COHORT_LIMIT,
)
if VALIDATION_RESULT.get("status") == "completed":
    WORKING_RESULT = run_m1_normalization_member(
        MSSV, drive_root=DRIVE_ROOT, local_root=LOCAL_ROOT, cohort_limit=COHORT_LIMIT,
    )
else:
    WORKING_RESULT = {"status": "chưa chạy", "reason": "Kiểm định chưa hoàn tất; hãy chạy lại ô này để tiếp tục."}
print({key: VALIDATION_RESULT.get(key) for key in (
    "status", "selection_fingerprint", "expected_id_count", "ready_for_m2",
    "ready_for_m3", "issues", "report_path",
)})
print("Bảng làm việc:", WORKING_RESULT)


## Dừng, tiếp tục và giới hạn kết quả

Thực hiện các bước theo thứ tự: arXiv, S2, kiểm định và tạo bảng làm việc. Nếu đã chạy S2 trước arXiv, hãy hoàn tất pha arXiv trước khi chạy kiểm định vì bước này cần archive nguồn hợp lệ. Khi một pha bị dừng, giữ nguyên MSSV, `COHORT_LIMIT` và Drive root rồi chạy lại ô tương ứng. Không để hai runtime cùng ghi một phạm vi.

Lượt 100 ID chỉ kiểm tra vận hành. Phép đo toàn nhóm cần bao phủ đủ 20.000 vị trí và ghi nhận thời gian chờ API; không suy rộng kết quả lượt thử ra toàn nhóm. Xem [hướng dẫn thành viên](../docs/M1_COLAB_MEMBER_GUIDE.md), [quy trình M1](../docs/operations/m1_production.md) và [hướng dẫn S2](../docs/guides/s2_integration.md).